In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-001.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-002.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-003.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-004.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-005.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-006.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-007.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dat

In [2]:
from pathlib import Path

input_dirs = list(Path("/kaggle/input").iterdir())
print("Attached inputs:", input_dirs)

pngs = sorted(Path("/kaggle/input").rglob("*.png"))
print("PNG count:", len(pngs))
for p in pngs[:10]:
  print(p)

Attached inputs: [PosixPath('/kaggle/input/datasets')]
PNG count: 11936
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-001.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-002.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-003.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-004.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-005.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-006.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-

In [3]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

data_root = Path('/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset')
png_paths = sorted(data_root.rglob('*.png'))

rows = []
for p in png_paths:
  seq_id = p.parent.name.lower()
  if seq_id.startswith('fall'):
    label = 1
    label_name = 'fall'
  elif seq_id.startswith('adl'):
    label = 0
    label_name = 'adl'
  else:
    continue

  rows.append({
      'path': str(p),
      'seq_id': seq_id,
      'label': label,
      'label_name': label_name,
  })

df = pd.DataFrame(rows)
print('Total PNGs:', len(df))
print('Unique sequences:', df['seq_id'].nunique())
print(df.groupby(['label_name', 'seq_id']).size().head(10))

# 1) Hold out 20% of sequences for test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(
    gss_test.split(df, df['label'], groups=df['seq_id'])
)
train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# 2) Split remaining sequences into ~85% train / 15% val
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(
    gss_val.split(
        train_val_df, train_val_df['label'], groups=train_val_df['seq_id']
    )
)
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

assert set(train_df['seq_id']).isdisjoint(val_df['seq_id'])
assert set(train_df['seq_id']).isdisjoint(test_df['seq_id'])
assert set(val_df['seq_id']).isdisjoint(test_df['seq_id'])

train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

manifest = pd.concat([train_df, val_df, test_df], ignore_index=True)
manifest.to_csv('/kaggle/working/ur_fall_manifest.csv', index=False)
print('Saved manifest to /kaggle/working/ur_fall_manifest.csv')
print(
    manifest.groupby('split').agg(
        frames=('path', 'count'), sequences=('seq_id', 'nunique')
    )
)

Total PNGs: 11936
Unique sequences: 70
label_name  seq_id         
adl         adl-01-cam0-rgb    150
            adl-02-cam0-rgb    180
            adl-03-cam0-rgb    180
            adl-04-cam0-rgb    150
            adl-05-cam0-rgb    180
            adl-06-cam0-rgb    230
            adl-07-cam0-rgb    180
            adl-08-cam0-rgb    180
            adl-09-cam0-rgb    150
            adl-10-cam0-rgb    300
dtype: int64
Saved manifest to /kaggle/working/ur_fall_manifest.csv
       frames  sequences
split                   
test     2653         14
train    7926         47
val      1357          9


In [4]:
import pandas as pd

manifest = pd.read_csv("/kaggle/working/ur_fall_manifest.csv")
print(
    pd.crosstab(
        manifest["split"], manifest["label_name"], margins=True, margins_name="Total"
    )
)
print(
    manifest.groupby(["split", "label_name"])["seq_id"]
    .nunique()
    .unstack(fill_value=0)
)

label_name   adl  fall  Total
split                        
test        2286   367   2653
train       5610  2316   7926
val         1045   312   1357
Total       8941  2995  11936
label_name  adl  fall
split                
test         10     4
train        25    22
val           5     4
